# Lab 2 — Exercise 6: SVM Classification of Landcover

Standalone notebook extracted from `761_Lab2_ans.ipynb` Exercise 6.

Shared Earth Engine / figure helpers live in `helpers/lab2_helpers.py` and can be imported by both this notebook and the main Lab 2 notebook.


# Set up


In [1]:
# Install libs if needed, note the --quiet option means a lot less annoying info than as Lab 1 when we did the same!
# !pip install geemap --quiet



In [2]:
# Set up GEE API
import ee
ee.Authenticate()
ee.Initialize(project='geog761-dongwook') #<- Remember to change this to your own project's name!


In [3]:
# Import other libs
import zipfile
import tempfile
import urllib.request

import geemap
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.patches as mpatches
from shapely.geometry import box
from sklearn.metrics import classification_report, f1_score

# Shared helpers
from helpers.lab2_helpers import (
    get_sentinel2,
    add_indices,
    fc_to_lists,
    get_bounds_coords,
    add_scale_bar,
    add_north_arrow,
    LCDB_NAMES,
    LCDB_PALETTE,
)


(6) Exercise: train an SVM model using the Landcare NZ 2018 landcover database record for the region. Producing a landcover map of Great Barrier (Aotea) Island for 2018 (based off the Austral summer of 18/19 S2).

Your map should be presented at a publication quality level with all the usual map components (scale, legend, north arrow, data attribution).

You will need to provide performance statistics of the model within your figure.

*   Here you can access the landcover database: https://lris.scinfo.org.nz/layer/104400-lcdb-v50-land-cover-database-version-50-mainland-new-zealand/. You will need to explore for yourself how to extract this data and then upload it to colab, then how to plug it into the SVM algorithim. I have provided some starter code below.

An intial workflow to get the data into the state you need it in to then use it as training data might look like:
- Download the ZIP manually from their browser, having set your area of interest and used the 'Export' tool top right.
- Upload it to Colab.
- Unzip it and load with GeoPandas.
(25 pts)



## Exercise 6: SVM Classification of Landcover


**HISTORY**
1. Baseline : Accuracy(0.5)+ macro F1 (0.09) , Random Sampling + numPixels:5000 (1455 samples)
2. Run 1 : Accuracy(0.32), Stratified Sampling + 300 numPoints
3. Run 2 : Accuracy(0.44), Run1 + Add more bands(features), Weak on spectrally similar woody/grass classes
4. Run 3 : Accuracy(0.52), Run2 + Add 'NDVI', 'NDWI', 'NBR', 'NDMI', 'EVI', 'Class_2018'
5. Run 4 : Accuracy(0.45), Run1 + only 'NDVI', 'NDWI', 'NBR', 'NDMI', 'EVI' indices

In [4]:
# Code to get you started
import zipfile
import geopandas as gpd

# Upload the ZIP manually using the Colab UI
# from google.colab import files
# uploaded = files.upload()  # <- Expects a ZIP

# Unzip
with zipfile.ZipFile("datasets/lris-lcdb-v60-land-cover-database-version-60-mainland-new-zealand-SHP.zip", 'r') as zip_ref: #<- Check file names
    zip_ref.extractall("datasets/lcdb")



In [5]:

# Read shapefile
gdf = gpd.read_file("datasets/lcdb/lcdb-v60-land-cover-database-version-60-mainland-new-zealand.shp") #<- Check file names


In [6]:
display(gdf.head(3))

,Name_2023,Name_2018,Name_2012,Name_2008,Name_2001,Name_1996,Class_2023,Class_2018,Class_2012,Class_2008,...,Onshore_23,Onshore_18,Onshore_12,Onshore_08,Onshore_01,Onshore_96,EditAuthor,EditDate,LCDB_UID,geometry
0,Not land,Sand and Gravel,Sand and Gravel,Sand and Gravel,Sand and Gravel,Sand and Gravel,0,10,10,10,...,no,yes,yes,yes,yes,yes,Landcare Research,2024-08-01,lcdb2000600005,"POLYGON ((168.90675 -43.89763, 168.90632 -43.8..."
1,Built-up Area (settlement),Built-up Area (settlement),Built-up Area (settlement),Built-up Area (settlement),Built-up Area (settlement),Built-up Area (settlement),1,1,1,1,...,yes,yes,yes,yes,yes,yes,Terralink,2004-06-30,lcdb2000000012,"POLYGON ((168.30432 -46.57797, 168.3023 -46.57..."
2,Built-up Area (settlement),Built-up Area (settlement),Built-up Area (settlement),Built-up Area (settlement),Built-up Area (settlement),Built-up Area (settlement),1,1,1,1,...,yes,yes,yes,yes,yes,yes,MfE (LUM),2013-07-01,lcdb2000001466,"POLYGON ((169.47548 -46.56152, 169.47547 -46.5..."


### 1. Fitering out 2024 landcover data

Select only 2023 data features along with the geometry.

In [7]:

columns = ['Name_2023','Class_2023','geometry']
gdf_filtered = gdf[columns]
display(gdf_filtered.head(3))


,Name_2023,Class_2023,geometry
0,Not land,0,"POLYGON ((168.90675 -43.89763, 168.90632 -43.8..."
1,Built-up Area (settlement),1,"POLYGON ((168.30432 -46.57797, 168.3023 -46.57..."
2,Built-up Area (settlement),1,"POLYGON ((169.47548 -46.56152, 169.47547 -46.5..."


### 2. Get Sentinel-2 data for 2023/24

Area of Interest is the rectangle area of Great Barrier Island.  
- **base_bands** : Baseline feature bands used for training  
- **additional_bands** : Additional feature bands to improve model performance  

In [8]:

# Producing a landcover map of Great Barrier (Aotea) Island for 2018 
aoi = ee.Geometry.Rectangle([175.28, -36.35, 175.55, -36.02])

# baseline bands
base_bands = ['B2', 'B3', 'B4', 'B8']

# feature engineering(bands)
additional_bands = [ 'B5', 'B6', 'B7', 'B8A', 'B11', 'B12']
bands = base_bands + additional_bands

# Class_2023 -> summer 2023/2024
s2_clipped = get_sentinel2('2023-12-01', '2024-02-28', aoi, bands)

# Add indices to the Sentinel-2 image, for more accurate classification
# (add_indices imported from helpers.lab2_helpers)
s2_clipped = add_indices(s2_clipped)


In [9]:
Map = geemap.Map(center=[-36.1953, 175.4372], zoom=10)
Map.addLayer(aoi, {}, 'AOI')
Map.addLayer(s2_clipped.select(base_bands), {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.2}, 'Sentinel-2 RGB')
# Visualize on the map to check got it right... (always a good idea)
Map

Map(center=[-36.1953, 175.4372], controls=(WidgetControl(options=['position', 'transparent_bg'], position='top…

### 3. Clip Landcover dataset

Clip the landcover dataset to the AOI

In [10]:
from shapely.geometry import box


# Convert Earth Engine geometry to shapely geometry for clipping
# Get the coordinates from the Earth Engine geometry
minx, miny = 175.28, -36.35
maxx, maxy = 175.55, -36.02

# Create the bounding box geometry
bbox_polygon = box(minx, miny, maxx, maxy)

# 3. Clip the GeoDataFrame
gdf_clipped = gdf_filtered.clip(bbox_polygon)

# Check the result
display(gdf_clipped.head())

,Name_2023,Class_2023,geometry
92149,Sand and Gravel,10,"POLYGON ((175.30228 -36.23857, 175.30192 -36.2..."
205004,Low Producing Grassland,41,"POLYGON ((175.30191 -36.2315, 175.30218 -36.23..."
301490,Manuka and/or Kanuka,52,"POLYGON ((175.30717 -36.23287, 175.30722 -36.2..."
301571,Manuka and/or Kanuka,52,"POLYGON ((175.302 -36.23397, 175.30177 -36.233..."
262561,Manuka and/or Kanuka,52,"POLYGON ((175.30341 -36.22954, 175.30352 -36.2..."


### 4. Preparing Training Data

1. Convert the geopandas dataframe to the gee FeatureCollection.(Vector --> Feature)
2. Convert ee.Image to ee.FeatureCollection dataset
3. Count the number of samples per class to check for class imbalance.


In [11]:


# Convert the GeoDataFrame to Earth Engine FeatureCollection using geemap
landcover_fc = geemap.gdf_to_ee(gdf_clipped)

# Create a raster from the landcover vectors using the 'Class' field
landcover_raster = landcover_fc.reduceToImage(
    properties=['Class_2023'],
    reducer=ee.Reducer.first()
).rename('Class')

# Add the landcover raster as a band to the Sentinel-2 image
training_data = s2_clipped.addBands(landcover_raster)

print("Available bands:", training_data.bandNames().getInfo())

Available bands: ['B2', 'B3', 'B4', 'B8', 'B5', 'B6', 'B7', 'B8A', 'B11', 'B12', 'NDVI', 'NDWI', 'NBR', 'NDMI', 'EVI', 'Class']


In [12]:
feature_bands = bands + ['NDVI', 'NDWI', 'NBR', 'NDMI', 'EVI']

# Original sampling code from lab2. 
# ex6_sample = training_data.select(bands + ['Class_2023']).sample( 
#     region=aoi,
#     scale=10,
#     numPixels=5000,
#     seed=2,
#     geometries=True)

# use stratifiedSampling for balanced sampling
ex6_sample = training_data.select(feature_bands + ['Class']).stratifiedSample(
    numPoints = 500,  # increase number of samples to prevent overfitting, initial : 100 with 2134 samples
    classBand='Class',
    region=aoi,
    scale=10, #10 m scale from each land cover class within a region of interest
    seed=2,
    geometries=True)

# Print first 10 sample points (client-side)
ex6_first_10 = ex6_sample.limit(10).getInfo()

print('\nFirst 10 training samples:')
for i, feature in enumerate(ex6_first_10['features']):
    props = feature['properties']
    # Print orignal bands and additional indices
    print(f"Sample {i+1}: Class={props['Class']}, B2={props.get('B2'):.2f}, B3={props.get('B3'):.2f}, B4={props.get('B4'):.2f}, B8={props.get('B8'):.2f}, \
        NDVI={props.get('NDVI'):.2f}, NDWI={props.get('NDWI'):.2f}, NBR={props.get('NBR'):.2f}, NDMI={props.get('NDMI'):.2f}, EVI={props.get('EVI'):.2f}")

# Count number of samples per class (server-side)
ex6_class_counts = ex6_sample.reduceColumns(
    reducer=ee.Reducer.frequencyHistogram(),
    selectors=['Class'])

print('\nClass distribution in sample:')
print(ex6_class_counts.getInfo())


First 10 training samples:
Sample 1: Class=1, B2=0.04, B3=0.07, B4=0.05, B8=0.44,         NDVI=0.81, NDWI=-0.71, NBR=0.64, NDMI=0.38, EVI=0.68
Sample 2: Class=1, B2=0.08, B3=0.10, B4=0.09, B8=0.39,         NDVI=0.61, NDWI=-0.58, NBR=0.52, NDMI=0.26, EVI=0.54
Sample 3: Class=1, B2=0.03, B3=0.05, B4=0.03, B8=0.38,         NDVI=0.85, NDWI=-0.75, NBR=0.63, NDMI=0.38, EVI=0.65
Sample 4: Class=1, B2=0.06, B3=0.08, B4=0.09, B8=0.27,         NDVI=0.50, NDWI=-0.52, NBR=0.46, NDMI=0.21, EVI=0.32
Sample 5: Class=1, B2=0.04, B3=0.07, B4=0.05, B8=0.32,         NDVI=0.74, NDWI=-0.66, NBR=0.50, NDMI=0.26, EVI=0.54
Sample 6: Class=1, B2=0.05, B3=0.07, B4=0.05, B8=0.32,         NDVI=0.72, NDWI=-0.64, NBR=0.43, NDMI=0.18, EVI=0.52
Sample 7: Class=1, B2=0.03, B3=0.05, B4=0.03, B8=0.31,         NDVI=0.83, NDWI=-0.74, NBR=0.71, NDMI=0.42, EVI=0.55
Sample 8: Class=1, B2=0.03, B3=0.05, B4=0.04, B8=0.32,         NDVI=0.78, NDWI=-0.72, NBR=0.58, NDMI=0.30, EVI=0.53
Sample 9: Class=1, B2=0.03, B3=0.05, B4=0.03

### 4. Split the training dataset to Train and Test
I am using only training and test datasets; validation dataset is not required as parameter tunning is not the outside the scope of this stage.

In [13]:
# Add random column
ex6_sample = ex6_sample.randomColumn('random',seed=2)

# Split
ex6_train = ex6_sample.filter(ee.Filter.lt('random', 0.8))
ex6_test = ex6_sample.filter(ee.Filter.gte('random', 0.8))

# number of samples
print(f"Number of total samples: {ex6_sample.size().getInfo()}")

Number of total samples: 9910


### 5. Training the SVM Model

1. Use the predefined hyperparameters in lab2 
2. Extract the ground-truth and predicted y-values from the trained model

In [14]:
# Define and train the SVM classifier
class_property = 'Class'
ex6_svm = ee.Classifier.libsvm(kernelType='RBF',gamma=0.5,cost=50).train(
    features=ex6_train,
    classProperty=class_property,
    inputProperties=feature_bands
)

# Classify validation and test sets
ex6_test_classified = ex6_test.classify(ex6_svm)



In [15]:
# return true and predicted values
y_true, y_pred = fc_to_lists(ex6_test_classified, 'Class', 'classification')
assert len(y_true) == len(y_pred), "y_true and y_pred have different lengths"


### 6. Evaludate the Performance of the SVM Model  

1. Fetch unique class names
2. Generate the error matrix from the trained classified model.
3. Print the confusion matrix, and generate the classification report using the scikit-learn API 


In [16]:
# order of classes in test dataset. I don't want to print all of them, which is not exist in the test dataset
ex6_order = ex6_test.aggregate_array('Class').distinct().sort() 

# use errorMatrix in google earth engine
ex6_test_matrix = ex6_test_classified.errorMatrix(class_property, 'classification',ex6_order)


In [17]:
# From lcdb-classes-at-version5.pdf (short names) — LCDB_NAMES imported from helpers

# List up only exist labels
labels = ex6_order.getInfo() 
names = [LCDB_NAMES.get(int(l), str(l))+'('+str(l)+')' for l in labels] 

print("Confusion Matrix:")
display(pd.DataFrame(ex6_test_matrix.getInfo(), index=names, columns=names))



Confusion Matrix:


,Built-up Area(1),Urban Parkland/Open Space(2),Transport Infrastructure(5),Surface Mine or Dump(6),Sand or Gravel(10),Landslide(12),Gravel or Rock(16),Lake or Pond(20),River(21),Estuarine Open Water(22),...,Low Producing Grassland(41),Herbaceous Freshwater Vegetation(45),Herbaceous Saline Vegetation(46),Gorse and/or Broom(51),Manuka and/or Kanuka(52),Broadleaved Indigenous Hardwoods(54),Forest - Harvested(64),Indigenous Forest(69),Mangrove(70),Exotic Forest(71)
Built-up Area(1),42,13,0,0,0,10,0,2,0,0,...,1,2,3,9,1,5,12,1,0,1
Urban Parkland/Open Space(2),14,58,0,0,0,6,0,0,0,0,...,0,0,0,0,1,0,8,0,0,0
Transport Infrastructure(5),0,0,64,0,0,4,0,0,0,0,...,0,0,0,2,1,0,3,0,0,0
Surface Mine or Dump(6),1,0,0,26,6,5,3,0,0,0,...,0,0,0,0,0,0,0,0,0,0
Sand or Gravel(10),5,0,0,2,23,15,26,0,1,4,...,1,0,4,0,1,2,5,0,1,0
Landslide(12),7,1,3,1,2,67,7,0,0,1,...,5,2,2,0,2,0,1,1,0,2
Gravel or Rock(16),3,2,0,0,3,19,72,0,1,1,...,0,0,1,0,1,0,2,0,0,2
Lake or Pond(20),1,0,0,0,1,0,0,29,0,0,...,0,0,0,0,0,0,0,0,0,0
River(21),0,0,0,1,0,0,0,0,28,0,...,0,0,0,0,0,0,0,0,0,0
Estuarine Open Water(22),0,0,0,0,1,0,0,0,1,59,...,0,7,13,0,0,0,0,0,15,0


In [18]:

report = classification_report(y_true, y_pred, labels=labels, target_names=names,output_dict=True)

In [19]:
ex6_macro_f1 = report["macro avg"]["f1-score"]
ex6_oa = report["accuracy"]

print(f"Original accuracy: {ex6_oa:.4f}")
print(f"Original macro F1: {ex6_macro_f1:.4f}")


Original accuracy: 0.5113
Original macro F1: 0.5249


### 7. Hyperparameter tuning (server-side on GEE)

❓ **THOUGHTS**  

Look at the confusion matrices between class 40 and 60, which represent vegetation and grassland, the trained model can't distinguish between classes, precision/recall is quite low compared to the rock,river and water.

One of my ideas to solve this problem is to group similar classes into a single class. For example, 'High Producing Exotic Grassland(40)' and 'Low Producing Exotic Grassland(41)' can be represented as 'Exotic Grassland'. 

Another approach is to generate more features. However, this approach requires a deep understading of spectral bands and indices.

The target size is too large compared to the input features. Try tuning the hyper parameter within the search space.  
- Example: [Spatial Thoughts GEE supplement](https://courses.spatialthoughts.com/end-to-end-gee-supplement.html#hyperparameter-tuning).

In [20]:
# Target Label
class_property = 'Class'

# Define the search space
cost_list = ee.List([1, 10, 50, 100])
gamma_list = ee.List([0.01, 0.1, 0.5, 1.0])

def evaluate_cost(cost):
    def evaluate_gamma(gamma):
        # Trainer
        classifier = ee.Classifier.libsvm(
            kernelType='RBF',
            cost=cost,
            gamma=gamma,
        ).train(
            features=ex6_train,
            classProperty=class_property,
            inputProperties=feature_bands,
        )
        # Evaluator
        accuracy = (
            ex6_test
            .classify(classifier)
            .errorMatrix(class_property, 'classification')
            .accuracy()
        )
        return ee.Feature(None, {
            'accuracy': accuracy,
            'cost': cost,
            'gamma': gamma,
        })

    return gamma_list.map(evaluate_gamma)

tuning_fc = ee.FeatureCollection(cost_list.map(evaluate_cost).flatten())

# Only this small FeatureCollection (16 rows) is brought client-side
tuning_rows = tuning_fc.sort('accuracy', False).getInfo()['features']
tuning_df = pd.DataFrame([f['properties'] for f in tuning_rows])
display(tuning_df)

best_row = tuning_df.iloc[0]
best_cost = float(best_row['cost'])
best_gamma = float(best_row['gamma'])
best_accuracy = float(best_row['accuracy'])

print(f'\nBest parameters: cost={best_cost:g}, gamma={best_gamma:g}')
print(f'Best validation accuracy: {best_accuracy:.4f}')


,accuracy,cost,gamma
0,0.561713,100,1.00
1,0.539043,50,1.00
2,0.526448,100,0.50
3,0.511335,50,0.50
4,0.499748,10,1.00
5,0.468010,100,0.10
6,0.460453,10,0.50
7,0.452393,50,0.10
8,0.416625,1,1.00
9,0.401008,10,0.10



Best parameters: cost=100, gamma=1
Best validation accuracy: 0.5617


In [21]:
# Retrain with the best GEE-tuned parameters and compare to the baseline SVM
ex6_svm_tuned = ee.Classifier.libsvm(
    kernelType='RBF',
    cost=best_cost,
    gamma=best_gamma,
).train(
    features=ex6_train,
    classProperty=class_property,
    inputProperties=feature_bands,
)

ex6_test_tuned = ex6_test.classify(ex6_svm_tuned)
y_true_tuned, y_pred_tuned = fc_to_lists(ex6_test_tuned, class_property, 'classification')

tuned_report = classification_report(
    y_true_tuned, y_pred_tuned, labels=labels, target_names=names, output_dict=True
)
tuned_accuracy = tuned_report['accuracy']
tuned_macro_f1 = tuned_report['macro avg']['f1-score']

print('Comparison with original model (gamma=0.5, cost=50):')
print(f'  Original accuracy: {ex6_oa:.4f}')
print(f'  Tuned accuracy:    {tuned_accuracy:.4f}  ({tuned_accuracy - ex6_oa:+.4f})')
print(f'  Original macro F1: {ex6_macro_f1:.4f}')
print(f'  Tuned macro F1:    {tuned_macro_f1:.4f}  ({tuned_macro_f1 - ex6_macro_f1:+.4f})')

print('\nTop parameter combinations (GEE validation accuracy):')
for _, row in tuning_df.head(10).iterrows():
    print(f"  {row['accuracy']:.4f}: cost={row['cost']:g}, gamma={row['gamma']:g}")


Comparison with original model (gamma=0.5, cost=50):
  Original accuracy: 0.5113
  Tuned accuracy:    0.5617  (+0.0504)
  Original macro F1: 0.5249
  Tuned macro F1:    0.5791  (+0.0542)

Top parameter combinations (GEE validation accuracy):
  0.5617: cost=100, gamma=1
  0.5390: cost=50, gamma=1
  0.5264: cost=100, gamma=0.5
  0.5113: cost=50, gamma=0.5
  0.4997: cost=10, gamma=1
  0.4680: cost=100, gamma=0.1
  0.4605: cost=10, gamma=0.5
  0.4524: cost=50, gamma=0.1
  0.4166: cost=1, gamma=1
  0.4010: cost=10, gamma=0.1


### 8. Figure map of the SVM model

Apply the trained SVM to the Great Barrier (Aotea) Island for 2025 (based off the Austral summer of 24/25) Sentinel-2 composite and inspect the classified map interactively, then present it as a publication-quality figure with a scale bar, north arrow, legend, data attribution and the model's performance statistics.

In [22]:
# 
s2_2025_clipped = get_sentinel2('2024-12-01', '2025-03-01', aoi, bands)
s2_2025_clipped = add_indices(s2_2025_clipped)

In [23]:
# Apply the tuned SVM to every pixel of the 2025 composite

ex6__2025_classified = s2_2025_clipped.select(feature_bands).classify(ex6_svm_tuned)
# Exclude the ocean pixels
land_mask = landcover_raster.select('Class').gt(0)
classified_land = ex6__2025_classified.updateMask(land_mask)

# Quick interactive look before committing to the publication figure
Map = geemap.Map()
Map.centerObject(aoi, 11)
Map.addLayer(s2_2025_clipped, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 0.3, 'gamma': 1.2}, 'Sentinel-2 RGB 2024')
Map.addLayer(classified_land.randomVisualizer(), {}, 'SVM landcover 2025')
Map.addLayer(aoi, {'color': 'red'}, 'AOI')
Map

Map(center=[-36.184959926082875, 175.41500000000002], controls=(WidgetControl(options=['position', 'transparen…

### 9. Publication-quality figure

- LCDB class codes are sparse (0, 1, 2, 5, 10, ... 80), so they are remapped onto 1...N before a palette can be applied. 
- Each code keeps a fixed colour so a class looks the same in every figure.
- Accuracy is calculated by tuned svm model. 

In [24]:
# Official LCDB v5.0 symbology colours — LCDB_PALETTE imported from helpers

# The SVM can only predict classes seen in the training split
ex6_classes = [int(c) for c in ex6_train.aggregate_array('Class').distinct().sort().getInfo()]
ex6_index = list(range(1, len(ex6_classes) + 1))
ex6_palette = [LCDB_PALETTE.get(c, '888888') for c in ex6_classes]
ex6_classified_vis = classified_land.remap(ex6_classes, ex6_index)
ex6_vis_params = {'min': 1, 'max': len(ex6_classes), 'palette': ex6_palette}


# Restrict the legend to classes that actually appear on the map
ex6_hist = classified_land.reduceRegion(
    reducer=ee.Reducer.frequencyHistogram(),
    geometry=aoi,
    scale=100,
    maxPixels=1e9,
).getInfo()['classification']
ex6_mapped = [c for c in ex6_classes if str(c) in ex6_hist or f'{c}.0' in ex6_hist]

# Thumbnails: true colour uses the same 0-0.3 reflectance stretch as the rest of the lab
ex6_xmin, ex6_xmax, ex6_ymin, ex6_ymax = get_bounds_coords(aoi)
ex6_thumb_params = {'dimensions': 600, 'region': aoi, 'format': 'png'} # hitting timeout on the high dimensions

ex6_rgb_url = s2_2025_clipped.visualize(
    bands=['B4', 'B3', 'B2'], min=0, max=0.3, gamma=1.2
).getThumbURL(ex6_thumb_params)
ex6_class_url = ex6_classified_vis.visualize(**ex6_vis_params).getThumbURL(ex6_thumb_params)



In [ ]:
# --- Figure layout: 2 maps + legend column 
fig = plt.figure(figsize=(15, 7.5))
gs = fig.add_gridspec(1, 3, width_ratios=[1, 1, 0.55], wspace=0.18)
ax_rgb = fig.add_subplot(gs[0, 0])
ax_class = fig.add_subplot(gs[0, 1])
ax_legend = fig.add_subplot(gs[0, 2])

panels = (
    (ax_rgb, ex6_rgb_url, f'(a) Sentinel-2 true colour'),
    (ax_class, ex6_class_url, f'(b) SVM landcover classification'),
)
for ax, url, title in panels:
    with tempfile.NamedTemporaryFile(suffix='.png') as f:
        urllib.request.urlretrieve(url, f.name)
        img = mpimg.imread(f.name)
        ax.imshow(img, extent=[ex6_xmin, ex6_xmax, ex6_ymin, ex6_ymax])
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Longitude', fontsize=10)
    ax.set_xlim(ex6_xmin, ex6_xmax)
    ax.set_ylim(ex6_ymin, ex6_ymax)
    ax.tick_params(labelsize=6)

ax_rgb.set_ylabel('Latitude', fontsize=10)

# Map furniture on the classified panel
add_scale_bar(ax_class, length_km=5)
add_north_arrow(ax_class)

# Legend column: only the classes present on the map
ax_legend.axis('off')
legend_patches = [
    mpatches.Patch(color=f'#{LCDB_PALETTE.get(c, "888888")}', label=f'{c}: {LCDB_NAMES.get(c, str(c))}')
    for c in ex6_mapped
]
ax_legend.legend(
    handles=legend_patches,
    loc='center left',
    ncol=1,
    fontsize=8,
    frameon=True,
    title='LCDB v5.0 class',
    title_fontsize=9,
)

fig.suptitle(
    'Great Barrier (Aotea) Island landcover from a Support Vector Machine, 2025\n'
    f'Sentinel-2 spectral bands and indices trained on LCDB v6.0 labels  |  '
    f'test overall accuracy {tuned_accuracy:.1%}, macro F1 {tuned_macro_f1:.2f}',
    fontsize=13,
    fontweight='bold',
)

fig.text(
    0.5, 0.0,
    'Figure 3. Landcover of Great Barrier (Aotea) Island classified with an RBF-kernel Support Vector Machine.\n'
    f'(a) Sentinel-2(2025), true colour, cloud masked.\n'
    f'(b) SVM prediction trained on Sentinel-2(2024) and LCDB v6.0 2024, samples using 10 spectral bands plus NDVI, NDWI, NBR, NDMI and EVI. \n'
    'The accuracy was tested using 20% of the LCDB data that wasn\'t used for training.',
    ha='center', fontsize=9, style='italic',
)

fig.text(
    0.1, -0.1,
    'Data: Copernicus Sentinel-2 MSI L2A (ESA/EU) via Google Earth Engine; '
    'Land Cover Database v6.0 (Manaaki Whenua - Landcare Research, LRIS). ',
    ha='left', fontsize=8, color='0.35',
)

plt.tight_layout()
fig.savefig('figures/lab2_ex6_aotea_svm_landcover_2025.png', dpi=300, bbox_inches='tight')
plt.show()